# Industry Expectations Across IT Career Clusters: Data Preparation and Cluster-Wise Exploratory Analysis

**Scope.** This notebook filters, cleans and analyses IT job postings for five career clusters: *Software Engineering*, *DevOps/Cloud Engineering*, *Cybersecurity*, *Networking & Infrastructure*, and *Data & AI Engineering*. Because the data contain no salary information, the analysis is built on job titles, cluster membership, required experience, job descriptions and the skills/technologies field.

**Pipeline**

1. Load the data, standardise column names and resolve the schema.
2. Assign each record to a career cluster and keep only the defined clusters.
3. Drop records with a missing cluster label or missing experience value (where an experience field exists).
4. Convert `experience_years` to numeric and remove IQR outliers within each cluster.
5. Tokenise the multi-value skills field and normalise skill names.
6. Produce four sets of charts (cluster size, experience, top skills per cluster, tool-frequency and specialisation), each saved as a 300 dpi PNG.

## 0. Data audit notes (read before running)

The dataset `IT_Job_Dataset_SriLanka_20000.csv` has **six columns**: `Job Role`, `Job Title`, `Student Qualification`, `Student Skills`, `Job Description`, and `experience_years`. Against the variables named in the research design:

| Variable in the design | Present in file | Handling in this notebook |
|---|---|---|
| career cluster label | no (only `Job Role`) | derived from `Job Role` through an explicit, editable mapping |
| job titles | yes (`Job Title`) | summarised per cluster |
| job descriptions | yes (`Job Description`) | term frequencies per cluster |
| required skills / technologies | `Student Skills` (skills held by a candidate profile) | used as the skills field; interpret as skills *observed* per role |
| `experience_years` | **yes** | cleaned via numeric parser, IQR outlier removal, and analysed in Chart 2 |
| posting date | **no** | the temporal part of Chart 4 runs only if a date column exists |

`experience_years` is fully integrated, enabling the complete cleaning pipeline, cluster-wise IQR outlier removal, distribution boxplots, and Kruskal-Wallis non-parametric hypothesis testing.


## 1. Setup and configuration

In [ ]:
import re
import textwrap
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from scipy import stats
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

warnings.filterwarnings("ignore")

# ----- paths -----------------------------------------------------------
DATA_PATH = Path("IT_Job_Dataset_SriLanka_20000.csv") if Path("IT_Job_Dataset_SriLanka_20000.csv").exists() else Path("../IT_Job_Dataset_SriLanka_20000.csv")
OUTPUT_DIR = Path("outputs/charts")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ----- figure settings -------------------------------------------------
DPI = 300
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# ----- research scope --------------------------------------------------
CLUSTERS = [
    "Software Engineering",
    "DevOps/Cloud Engineering",
    "Cybersecurity",
    "Networking & Infrastructure",
    "Data & AI Engineering",
]
CLUSTER_COLORS = dict(zip(CLUSTERS, sns.color_palette("colorblind", len(CLUSTERS))))

# Used only when the file has no explicit cluster column.
# Roles absent from this mapping are treated as outside the research scope.
ROLE_TO_CLUSTER = {
    "software engineer":          "Software Engineering",
    "web developer":              "Software Engineering",
    "mobile app developer":       "Software Engineering",
    "devops engineer":            "DevOps/Cloud Engineering",
    "cybersecurity analyst":      "Cybersecurity",
    "network administrator":      "Networking & Infrastructure",
    "it support specialist":      "Networking & Infrastructure",
    "data analyst":               "Data & AI Engineering",
    "machine learning engineer":  "Data & AI Engineering",
}

# Spellings of cluster labels that may appear in an explicit cluster column.
CLUSTER_LABEL_ALIASES = {
    "softwareengineering": "Software Engineering",
    "devopscloudengineering": "DevOps/Cloud Engineering",
    "devopscloud": "DevOps/Cloud Engineering",
    "devops": "DevOps/Cloud Engineering",
    "cloudengineering": "DevOps/Cloud Engineering",
    "cybersecurity": "Cybersecurity",
    "networkinginfrastructure": "Networking & Infrastructure",
    "networkinfrastructure": "Networking & Infrastructure",
    "networkingandinfrastructure": "Networking & Infrastructure",
    "dataaiengineering": "Data & AI Engineering",
    "dataandaiengineering": "Data & AI Engineering",
    "dataai": "Data & AI Engineering",
}

# Candidate column names (after snake_case standardisation), in priority order.
COLUMN_ALIASES = {
    "cluster":    ["career_cluster", "cluster", "it_cluster", "job_cluster"],
    "role":       ["job_role", "role", "job_title", "title"],
    "experience": ["experience_years", "required_experience", "experience", "years_of_experience"],
    "description": ["job_description", "description", "job_summary"],
    "date":       ["posted_date", "posting_date", "date_posted", "date", "year"],
    "skills":     ["required_skills", "skills", "required_technologies", "technologies", "student_skills"],
}

# Surface forms of the same skill that should be merged.
SKILL_ALIASES = {
    "nodejs": "Node.js", "node": "Node.js", "node js": "Node.js", "node.js": "Node.js",
    "powerbi": "Power BI", "power-bi": "Power BI", "power bi": "Power BI",
    "js": "JavaScript", "javascript": "JavaScript",
    "k8s": "Kubernetes",
    "c sharp": "C#", "csharp": "C#",
    "amazon web services": "AWS",
    "reactjs": "React", "react.js": "React",
    "angularjs": "Angular",
    "ms excel": "Excel", "microsoft excel": "Excel",
    "postgres": "PostgreSQL",
}

# Grouping of tools used in Chart 4 (unlisted tools fall under "Other").
TOOL_CATEGORIES = {
    "Python": "Programming languages", "Java": "Programming languages",
    "JavaScript": "Programming languages", "C#": "Programming languages",
    "SQL": "Data & analytics",
    "Excel": "Data & analytics", "Power BI": "Data & analytics",
    "HTML": "Web technologies", "CSS": "Web technologies", "React": "Web technologies",
    "Angular": "Web technologies", "Node.js": "Web technologies",
    "Docker": "Cloud & DevOps", "Kubernetes": "Cloud & DevOps",
    "AWS": "Cloud & DevOps", "Linux": "Cloud & DevOps",
}
ENTRY_MAX_YEARS = 2     # experience <= 2 years -> entry level
SENIOR_MIN_YEARS = 5    # experience >= 5 years -> senior
TOP_N = 5

IQR_K = 1.5              # Tukey fence multiplier
DROP_DUPLICATES = False  # postings carry no unique identifier, so exact duplicates cannot be told apart from distinct postings

## 2. Data loading and schema resolution

In [ ]:
def to_snake(name: str) -> str:
    return re.sub(r"[^0-9a-z]+", "_", str(name).strip().lower()).strip("_")


def resolve(df: pd.DataFrame, key: str):
    """Return the first column of `df` matching the aliases registered for `key`."""
    for candidate in COLUMN_ALIASES[key]:
        if candidate in df.columns:
            return candidate
    return None


raw = pd.read_csv(DATA_PATH)
raw.columns = [to_snake(c) for c in raw.columns]

print(f"Records: {len(raw):,} | Columns: {raw.shape[1]}")
print("Columns:", list(raw.columns))
print(f"Exact duplicate rows: {raw.duplicated().sum():,}")
display(raw.head())
display(raw.isna().sum().rename("missing_values").to_frame())

In [ ]:
cluster_col = resolve(raw, "cluster")
role_col    = resolve(raw, "role")
exp_col     = resolve(raw, "experience")
desc_col    = resolve(raw, "description")
date_col    = resolve(raw, "date")
skills_col  = resolve(raw, "skills")

schema = pd.Series({
    "cluster": cluster_col, "role": role_col, "experience": exp_col,
    "description": desc_col, "date": date_col, "skills": skills_col,
}, name="resolved_column").fillna("NOT FOUND")
display(schema.to_frame())

HAS_EXPERIENCE = exp_col is not None
HAS_DATE = date_col is not None

if skills_col is None:
    raise ValueError("No skills column found; add the column name to COLUMN_ALIASES['skills'].")
if cluster_col is None and role_col is None:
    raise ValueError("Neither a cluster nor a role column was found.")

if not HAS_EXPERIENCE:
    print("NOTE: no experience column found -> experience cleaning and Chart 2 will be skipped.")
if not HAS_DATE:
    print("NOTE: no posting-date column found -> the temporal panel of Chart 4 will be skipped.")

## 3. Cluster-specific filtering

Each record is assigned to one of the five defined clusters, either from an explicit cluster column or, when none exists, from `Job Role` via `ROLE_TO_CLUSTER`. Records whose cluster is missing or outside the defined set are removed. Rows with a missing experience value are then dropped, if an experience field exists. A row-count ledger is kept so every exclusion is traceable.

In [ ]:
df = raw.copy()
ledger = [("Raw records", len(df))]

if DROP_DUPLICATES:
    df = df.drop_duplicates()
    ledger.append(("After removing exact duplicates", len(df)))

def label_key(x):
    return re.sub(r"[^a-z]", "", str(x).lower())

# --- assign career cluster ---------------------------------------------
if cluster_col is not None:
    source = df[cluster_col]
    df["career_cluster"] = source.map(
        lambda v: CLUSTER_LABEL_ALIASES.get(label_key(v)) if pd.notna(v) else np.nan
    )
    n_missing = int(source.isna().sum())
    cluster_source = f"explicit column '{cluster_col}'"
else:
    source = df[role_col]
    df["career_cluster"] = source.map(
        lambda v: ROLE_TO_CLUSTER.get(str(v).strip().lower()) if pd.notna(v) else np.nan
    )
    n_missing = int(source.isna().sum())
    cluster_source = f"derived from '{role_col}' via ROLE_TO_CLUSTER"
print("Cluster assignment:", cluster_source)

# --- drop missing cluster, then out-of-scope ------------------------
df = df[source.notna().reindex(df.index)]
ledger.append(("After dropping missing cluster/role label", len(df)))

out_of_scope = df["career_cluster"].isna()
print("Out-of-scope categories removed:")
print(df.loc[out_of_scope, cluster_col or role_col].value_counts().to_string() or "  none")
df = df[~out_of_scope].copy()
ledger.append(("After keeping the five defined clusters", len(df)))

# --- drop missing experience (only where the field exists) ------------
if HAS_EXPERIENCE:
    df = df[df[exp_col].notna()]
    ledger.append(("After dropping missing experience", len(df)))

df["career_cluster"] = pd.Categorical(df["career_cluster"], categories=CLUSTERS, ordered=True)

## 4. Numeric conversion and IQR outlier removal

`experience_years` is converted to a number. Text such as `"2-4 years"` or `"5+ years"` is parsed, ranges are replaced by their midpoint, and negative values are treated as invalid.

Outliers are identified with Tukey's fences (Q1 − 1.5·IQR, Q3 + 1.5·IQR) computed **within each cluster**. Clusters differ in seniority profile, so a pooled fence would disproportionately flag legitimate observations from the more senior clusters.

In [ ]:
def parse_numeric(value):
    """Convert messy numeric text to a float; ranges become midpoints."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).lower().replace(",", "")
    text = re.sub(r"(?<=\d)\s*k\b", "000", text)          # 150k -> 150000
    numbers = [float(n) for n in re.findall(r"\d+(?:\.\d+)?", text)]
    return float(np.mean(numbers[:2])) if numbers else np.nan


def iqr_bounds(series: pd.Series, k: float = IQR_K):
    q1, q3 = series.quantile([0.25, 0.75])
    spread = q3 - q1
    return q1 - k * spread, q3 + k * spread


def remove_iqr_outliers(frame: pd.DataFrame, columns, group_col="career_cluster", k=IQR_K):
    keep = pd.Series(True, index=frame.index)
    records = []
    for group, sub in frame.groupby(group_col, observed=True):
        for col in columns:
            lo, hi = iqr_bounds(sub[col], k)
            flagged = (sub[col] < lo) | (sub[col] > hi)
            keep.loc[sub.index[flagged]] = False
            records.append({"cluster": group, "variable": col, "lower_fence": round(lo, 2),
                            "upper_fence": round(hi, 2), "outliers": int(flagged.sum()), "n": len(sub)})
    return frame[keep].copy(), pd.DataFrame(records)


numeric_cols = []
if HAS_EXPERIENCE:
    df["experience_years"] = df[exp_col].map(parse_numeric)
    df.loc[df["experience_years"] < 0, "experience_years"] = np.nan
    numeric_cols.append("experience_years")

if numeric_cols:
    df = df.dropna(subset=numeric_cols)
    ledger.append(("After numeric conversion (unparseable/invalid removed)", len(df)))
    df, outlier_log = remove_iqr_outliers(df, numeric_cols)
    ledger.append(("After IQR outlier removal", len(df)))
    display(outlier_log)
else:
    print("No continuous variables available -> numeric conversion and IQR step skipped.")

## 5. Tokenisation of the multi-value skills field

Skills are split on commas, semicolons and pipes (slashes are preserved so that entries such as `CI/CD` stay intact). Tokens are trimmed, whitespace is normalised, and spelling variants are merged through `SKILL_ALIASES` and a most-frequent-surface-form rule. Duplicate skills within one posting are counted once, so frequencies are the **share of postings in a cluster that mention a skill**.

In [ ]:
def split_skills(cell):
    if pd.isna(cell):
        return []
    tokens = []
    for part in re.split(r"[,;|\n]+", str(cell)):
        part = re.sub(r"\s+", " ", part).strip(" .\t\"'[]()")
        if part:
            tokens.append(part)
    return tokens


df["_raw_tokens"] = df[skills_col].map(split_skills)

# most frequent surface form per case-insensitive key, then explicit aliases on top
surface = pd.Series([t for toks in df["_raw_tokens"] for t in toks])
canon = (surface.groupby(surface.str.lower())
                .agg(lambda s: s.value_counts().idxmax())
                .to_dict())
canon.update({k.lower(): v for k, v in SKILL_ALIASES.items()})

df["skills_list"] = df["_raw_tokens"].map(
    lambda toks: sorted({canon.get(t.lower(), t) for t in toks})
)
df["n_skills"] = df["skills_list"].map(len)

skills_long = (df[["career_cluster", "skills_list"]]
               .explode("skills_list")
               .dropna(subset=["skills_list"])
               .rename(columns={"skills_list": "skill"}))

postings_with_skills = (df[df["n_skills"] > 0]
                        .groupby("career_cluster", observed=True).size())
skill_counts = (skills_long.groupby(["career_cluster", "skill"], observed=True)
                .size().rename("postings").reset_index())
cluster_totals = {str(k): int(v) for k, v in postings_with_skills.items()}
skill_counts["career_cluster"] = skill_counts["career_cluster"].astype(str)
skill_counts["share_pct"] = (skill_counts["postings"]
                             / skill_counts["career_cluster"].map(cluster_totals) * 100)

print(f"Distinct skills after normalisation: {skills_long['skill'].nunique()}")
print(f"Skills per posting: mean = {df['n_skills'].mean():.2f}, "
      f"min = {df['n_skills'].min()}, max = {df['n_skills'].max()}")
print(f"Postings with no parsable skills: {(df['n_skills'] == 0).sum()}")

### Data-preparation ledger

In [ ]:
ledger_df = pd.DataFrame(ledger, columns=["step", "records_remaining"])
ledger_df["removed_in_step"] = (-ledger_df["records_remaining"].diff()).fillna(0).astype(int)
display(ledger_df)
ledger_df.to_csv(OUTPUT_DIR.parent / "data_preparation_ledger.csv", index=False)

df.drop(columns=["_raw_tokens"]).assign(skills_list=df["skills_list"].map("; ".join)) \
  .to_csv(OUTPUT_DIR.parent / "cleaned_cluster_dataset.csv", index=False)
print(f"Analysis sample: {len(df):,} records across {df['career_cluster'].nunique()} clusters")